# EDA — Franchise A (Brick-and-Mortar Garment Stores)

**Data format:** CSV (UTF-8 BOM) · Monthly sales files · Store master · Product master + Customer master from manufacturer

**Key traits:** Uses manufacturer's SKU codes directly · References loyalty customer IDs (up to ~21% blank = walk-in) · Returns encoded as negative quantities · ~0.5% deliberate data-quality errors

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

BASE = os.path.join('..', 'data_generator', 'storage', 'landing_zone')

## 1 · Load Data

In [ ]:
# Master data
product = pd.read_csv(os.path.join(BASE, 'master_company', 'product_master.csv'), encoding='utf-8-sig')
customer = pd.read_csv(os.path.join(BASE, 'master_company', 'customer_master.csv'), encoding='utf-8-sig')
stores = pd.read_csv(os.path.join(BASE, 'franchise_a', 'storemaster_monthly_refresh.csv'), encoding='utf-8-sig')

# Concatenate all monthly + weekly sales files
sales_files = sorted(glob.glob(os.path.join(BASE, 'franchise_a', 'sales_*.csv')))
print(f'Sales files found: {len(sales_files)}')
for f in sales_files:
    print(f'  {os.path.basename(f)}')

sales = pd.concat(
    [pd.read_csv(f, encoding='utf-8-sig') for f in sales_files],
    ignore_index=True
)
sales['transaction_timestamp'] = pd.to_datetime(sales['transaction_timestamp'])
print(f'\nTotal sales rows: {len(sales):,}')

## 2 · Schema & Basic Profiling

In [ ]:
print('=== Sales ===')
display(sales.info())
display(sales.describe())
display(sales.head())

In [ ]:
print('=== Product Master ===')
display(product.info())
display(product.head())

print('\n=== Customer Master ===')
display(customer.info())
display(customer.head())

print('\n=== Store Master ===')
display(stores.info())
display(stores.head())

In [ ]:
# Null counts
print('Null counts — Sales')
display(sales.isnull().sum())

print('\nNull counts — Product Master')
display(product.isnull().sum())

## 3 · Data Quality Checks (Quarantine Rules)

| Rule | Description |
|------|-------------|
| A1 | `unit_price` must not be NaN or ≤ 0 |
| A2 | `quantity` must be reasonable (abs > 100 is suspicious) |
| A3 | `transaction_timestamp` cannot be in the future |
| A4 | `sku_code` must exist in product master |

In [ ]:
now = pd.Timestamp.now()
valid_skus = set(product['sku_code'])

qa = pd.DataFrame({
    'A1_bad_price': sales['unit_price'].isna() | (sales['unit_price'] <= 0),
    'A2_extreme_qty': sales['quantity'].abs() > 100,
    'A3_future_date': sales['transaction_timestamp'] > now,
    'A4_unknown_sku': ~sales['sku_code'].isin(valid_skus),
})
qa['any_error'] = qa.any(axis=1)

print('=== Quarantine Summary ===')
display(qa.sum())
print(f'\nError rate: {qa["any_error"].mean():.4%}')

In [ ]:
# Sample quarantined rows
display(sales[qa['any_error']].head(10))

In [ ]:
# Separate clean data for downstream analysis
clean = sales[~qa['any_error']].copy()
print(f'Clean rows: {len(clean):,} / {len(sales):,}')

## 4 · Enrich with Product Master

In [ ]:
enriched = clean.merge(product[['sku_code', 'brand', 'category', 'sub_category', 'size', 'color', 'fabric', 'mrp']],
                       on='sku_code', how='left')
enriched['revenue'] = enriched['quantity'] * enriched['unit_price']
enriched['month'] = enriched['transaction_timestamp'].dt.to_period('M')
display(enriched.head())

## 5 · Sales Analysis

In [ ]:
# Monthly revenue trend
monthly = enriched.groupby('month')['revenue'].sum()
ax = monthly.plot(kind='bar', figsize=(12, 4), title='Monthly Revenue — Franchise A')
ax.set_ylabel('Revenue (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by category
cat_rev = enriched.groupby('category')['revenue'].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cat_rev.plot(kind='bar', ax=axes[0], title='Revenue by Category')
cat_rev.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', title='Category Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 stores by revenue
store_rev = (enriched.merge(stores, on='store_code', how='left')
             .groupby(['store_code', 'store_name', 'city'])['revenue'].sum()
             .sort_values(ascending=False).head(10))
display(store_rev.reset_index())

In [ ]:
# Top sizes and colors by units sold
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
enriched.groupby('size')['quantity'].sum().sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[0], title='Top 10 Sizes by Units Sold')
enriched.groupby('color')['quantity'].sum().sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[1], title='Top 10 Colors by Units Sold')
plt.tight_layout()
plt.show()

## 6 · Returns Analysis

In [ ]:
returns = enriched[enriched['transaction_type'] == 'RETURN'].copy()
normal_sales = enriched[enriched['transaction_type'] == 'SALE'].copy()

print(f'Sales rows : {len(normal_sales):,}')
print(f'Return rows: {len(returns):,}')
print(f'Return rate: {len(returns) / len(enriched):.2%}')

# Returns by category
if len(returns):
    returns.groupby('category')['quantity'].sum().abs().sort_values(ascending=False).plot(
        kind='bar', figsize=(8, 3), title='Return Units by Category')
    plt.tight_layout()
    plt.show()

## 7 · Loyalty vs Walk-in Analysis

In [ ]:
enriched['is_loyalty'] = enriched['customer_id'].astype(str).str.strip().ne('')
loyalty_split = enriched.groupby('is_loyalty')['revenue'].sum()
loyalty_split.index = loyalty_split.index.map({True: 'Loyalty', False: 'Walk-in'})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
loyalty_split.plot(kind='pie', ax=axes[0], autopct='%1.1f%%', title='Revenue: Loyalty vs Walk-in')
axes[0].set_ylabel('')

# Loyalty tier breakdown
loyalty_rows = enriched[enriched['is_loyalty']].merge(customer[['customer_id', 'loyalty_tier']], on='customer_id', how='left')
loyalty_rows.groupby('loyalty_tier')['revenue'].sum().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], title='Revenue by Loyalty Tier')
plt.tight_layout()
plt.show()

print(f'Walk-in (blank customer_id) rate: {(~enriched["is_loyalty"]).mean():.2%}')

## 8 · Hindi / Unicode Check

In [ ]:
# Detect Devanagari characters (U+0900–U+097F)
hindi_re = r'[\u0900-\u097F]'

hindi_products = product[product['product_name'].str.contains(hindi_re, na=False)]
hindi_stores = stores[stores['store_name'].str.contains(hindi_re, na=False)]
hindi_customers = customer[customer['customer_name'].str.contains(hindi_re, na=False)]

print(f'Hindi product names : {len(hindi_products)} / {len(product)} ({len(hindi_products)/len(product):.1%})')
print(f'Hindi store names   : {len(hindi_stores)} / {len(stores)} ({len(hindi_stores)/len(stores):.1%})')
print(f'Hindi customer names: {len(hindi_customers)} / {len(customer)} ({len(hindi_customers)/len(customer):.1%})')

display(hindi_products[['sku_code', 'product_name']].head())
display(hindi_stores.head())

## 9 · Summary

| Metric | Value |
|--------|-------|
| Total rows loaded | see above |
| Quarantine error rate | ~0.5% |
| Walk-in (no loyalty) rate | ~15-21% |
| Hindi text present | ✅ verified |
| Key errors found | NaN prices, qty=-500, future dates (2030) |